## Dataset and feature engineering

Load the ipl_macthes data first

In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

df=pd.read_csv("../data/raw/ipl_matches_data.csv")
df.head()

,match_id,season_id,balls_per_over,city,match_date,event_name,match_number,gender,match_type,format,overs,season,team_type,venue,toss_winner,team1,team2,toss_decision,match_winner,win_by_runs,win_by_wickets,result,player_of_match
0,335982,2008,6,Bangalore,2008-04-18,Indian Premier League,1.0,male,T20,T20,20,2008,club,M Chinnaswamy Stadium,1,1,6,field,6,140.0,NaN,win,46.0
1,1082591,2017,6,Hyderabad,2017-04-05,Indian Premier League,1.0,male,T20,T20,20,2017,club,"Rajiv Gandhi International Stadium, Uppal",1,2,1,field,2,35.0,NaN,win,15.0
2,1082592,2017,6,Pune,2017-04-06,Indian Premier League,2.0,male,T20,T20,20,2017,club,Maharashtra Cricket Association Stadium,4,4,3,field,4,NaN,7.0,win,36.0
3,1082593,2017,6,Rajkot,2017-04-07,Indian Premier League,3.0,male,T20,T20,20,2017,club,Saurashtra Cricket Association Stadium,6,5,6,field,6,NaN,10.0,win,57.0
4,1082594,2017,6,Indore,2017-04-08,Indian Premier League,4.0,male,T20,T20,20,2017,club,Holkar Cricket Stadium,494,494,4,field,494,NaN,6.0,win,71.0


Select the relevant columns from the above dataset

In [2]:
relevant_columns = [
    "match_id",
    "season_id",
    "match_date",
    "city",
    "venue",
    "toss_winner",
    "team1",
    "team2",
    "toss_decision",
    "match_winner",
    "result",
]

df=df[relevant_columns]
df

,match_id,season_id,match_date,city,venue,toss_winner,team1,team2,toss_decision,match_winner,result
0,335982,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win
1,1082591,2017,2017-04-05,Hyderabad,"Rajiv Gandhi International Stadium, Uppal",1,2,1,field,2,win
2,1082592,2017,2017-04-06,Pune,Maharashtra Cricket Association Stadium,4,4,3,field,4,win
3,1082593,2017,2017-04-07,Rajkot,Saurashtra Cricket Association Stadium,6,5,6,field,6,win
4,1082594,2017,2017-04-08,Indore,Holkar Cricket Stadium,494,494,4,field,494,win
...,...,...,...,...,...,...,...,...,...,...,...
1207,1529282,2026,2026-04-27,Delhi,"Arun Jaitley Stadium, Delhi",1,252,1,field,1,win
1208,1529283,2026,2026-04-28,New Chandigarh,Maharaja Yadavindra Singh International Cricke...,134,494,134,field,134,win
1209,1529284,2026,2026-04-29,Mumbai,"Wankhede Stadium, Mumbai",3,3,2,bat,2,win
1210,1529285,2026,2026-04-30,Ahmedabad,"Narendra Modi Stadium, Ahmedabad",615,1,615,field,615,win


Now, we first calculate the total runs from the ball by ball dataset to calculate whats the total target a team is chasing i.e calculate the total score for 1st innings. And then merge them into the above reduced dataframe

In [3]:
df2=pd.read_csv("../data/raw/ball_by_ball_data.csv")

first_innings=(
    df2[df2["innings"]==1]
    .groupby("match_id")["total_runs"]
    .sum()
    .reset_index()
)

first_innings

,match_id,total_runs
0,335982,222
1,335983,240
2,335984,129
3,335985,165
4,335986,110
...,...,...
1207,1529282,75
1208,1529283,222
1209,1529284,243
1210,1529285,155


Then merge the total scores onto the reduced ipl_matches dataframe

In [4]:
df = df.merge(
    first_innings.rename(columns={"total_runs": "first_innings_runs"}),
    on="match_id",
    how="left"
)
df.head()

,match_id,season_id,match_date,city,venue,toss_winner,team1,team2,toss_decision,match_winner,result,first_innings_runs
0,335982,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222
1,1082591,2017,2017-04-05,Hyderabad,"Rajiv Gandhi International Stadium, Uppal",1,2,1,field,2,win,207
2,1082592,2017,2017-04-06,Pune,Maharashtra Cricket Association Stadium,4,4,3,field,4,win,184
3,1082593,2017,2017-04-07,Rajkot,Saurashtra Cricket Association Stadium,6,5,6,field,6,win,183
4,1082594,2017,2017-04-08,Indore,Holkar Cricket Stadium,494,494,4,field,494,win,163


### We now have to convert this data into the ball by ball data form but to feed it into the ML model, we need cumulative values not individual batsman or over wise scores. We need need to construct some of the features from the ball by ball dataset like CRR, RRR

First filter the ball by ball dataset for 2nd innings score

In [5]:
test=df2[(df2["innings"]==2) & (df2["is_super_over"]==True)]
# checking if super overs are also present as 2nd innings, they are present as different innings so we can safely filter them with innings =2

second_innings=df2[df2["innings"]==2]
second_innings.head()

,season_id,match_id,batter,bowler,non_striker,team_batting,team_bowling,over_number,ball_number,batter_runs,extras,total_runs,batsman_type,bowler_type,player_out,fielders_involved,is_wicket,is_wide_ball,is_no_ball,is_leg_bye,is_bye,is_penalty,wide_ball_runs,no_ball_runs,leg_bye_runs,bye_runs,penalty_runs,wicket_kind,is_super_over,innings
124,2008,335982,R Dravid,AB Dinda,W Jaffer,1,6,0,0,1,0,1,Right hand Bat,Right arm Medium fast,NaN,NaN,False,False,False,False,False,False,0,0,0,0,0,NaN,False,2
125,2008,335982,W Jaffer,AB Dinda,R Dravid,1,6,0,1,0,1,1,Right hand Bat,Right arm Medium fast,NaN,NaN,False,True,False,False,False,False,1,0,0,0,0,NaN,False,2
126,2008,335982,W Jaffer,AB Dinda,R Dravid,1,6,0,2,0,0,0,Right hand Bat,Right arm Medium fast,NaN,NaN,False,False,False,False,False,False,0,0,0,0,0,NaN,False,2
127,2008,335982,W Jaffer,AB Dinda,R Dravid,1,6,0,3,1,0,1,Right hand Bat,Right arm Medium fast,NaN,NaN,False,False,False,False,False,False,0,0,0,0,0,NaN,False,2
128,2008,335982,R Dravid,AB Dinda,W Jaffer,1,6,0,4,1,0,1,Right hand Bat,Right arm Medium fast,NaN,NaN,False,False,False,False,False,False,0,0,0,0,0,NaN,False,2


Calculate the cumulative score each team is scoring after every ball as well as the wickets

In [6]:
second_innings["current_score"]=second_innings.groupby("match_id")["total_runs"].cumsum()
second_innings["wickets_lost"]=second_innings.groupby("match_id")["is_wicket"].cumsum()
second_innings.head()

,season_id,match_id,batter,bowler,non_striker,team_batting,team_bowling,over_number,ball_number,batter_runs,extras,total_runs,batsman_type,bowler_type,player_out,fielders_involved,is_wicket,is_wide_ball,is_no_ball,is_leg_bye,is_bye,is_penalty,wide_ball_runs,no_ball_runs,leg_bye_runs,bye_runs,penalty_runs,wicket_kind,is_super_over,innings,current_score,wickets_lost
124,2008,335982,R Dravid,AB Dinda,W Jaffer,1,6,0,0,1,0,1,Right hand Bat,Right arm Medium fast,NaN,NaN,False,False,False,False,False,False,0,0,0,0,0,NaN,False,2,1,0
125,2008,335982,W Jaffer,AB Dinda,R Dravid,1,6,0,1,0,1,1,Right hand Bat,Right arm Medium fast,NaN,NaN,False,True,False,False,False,False,1,0,0,0,0,NaN,False,2,2,0
126,2008,335982,W Jaffer,AB Dinda,R Dravid,1,6,0,2,0,0,0,Right hand Bat,Right arm Medium fast,NaN,NaN,False,False,False,False,False,False,0,0,0,0,0,NaN,False,2,2,0
127,2008,335982,W Jaffer,AB Dinda,R Dravid,1,6,0,3,1,0,1,Right hand Bat,Right arm Medium fast,NaN,NaN,False,False,False,False,False,False,0,0,0,0,0,NaN,False,2,3,0
128,2008,335982,R Dravid,AB Dinda,W Jaffer,1,6,0,4,1,0,1,Right hand Bat,Right arm Medium fast,NaN,NaN,False,False,False,False,False,False,0,0,0,0,0,NaN,False,2,4,0


Now reduce the dataset and select only relevant features

In [7]:
second_innings = second_innings[
    [
        "match_id",
        "innings",
        "team_batting",
        "team_bowling",
        "over_number",
        "ball_number",
        "total_runs",
        "is_wicket",
        "is_wide_ball",
        "is_no_ball",
        "current_score",
        "wickets_lost"
    ]
]

second_innings

,match_id,innings,team_batting,team_bowling,over_number,ball_number,total_runs,is_wicket,is_wide_ball,is_no_ball,current_score,wickets_lost
124,335982,2,1,6,0,0,1,False,False,False,1,0
125,335982,2,1,6,0,1,1,False,True,False,2,0
126,335982,2,1,6,0,2,0,False,False,False,2,0
127,335982,2,1,6,0,3,1,False,False,False,3,0
128,335982,2,1,6,0,4,1,False,False,False,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...
288221,1529286,2,252,134,18,3,1,False,False,False,216,3
288222,1529286,2,252,134,18,4,1,False,False,False,217,3
288223,1529286,2,252,134,18,5,4,False,False,False,221,3
288224,1529286,2,252,134,19,0,4,False,False,False,225,3


Now merge the initial matches dataset with this dataset to create a whole ball by ball dataset with total score scored in the 1st innnings in it

In [8]:
second_innings_merged = second_innings.merge(
    df,
    on="match_id",
    how="left"
)

second_innings_merged

,match_id,innings,team_batting,team_bowling,over_number,ball_number,total_runs,is_wicket,is_wide_ball,is_no_ball,current_score,wickets_lost,season_id,match_date,city,venue,toss_winner,team1,team2,toss_decision,match_winner,result,first_innings_runs
0,335982,2,1,6,0,0,1,False,False,False,1,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222
1,335982,2,1,6,0,1,1,False,True,False,2,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222
2,335982,2,1,6,0,2,0,False,False,False,2,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222
3,335982,2,1,6,0,3,1,False,False,False,3,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222
4,335982,2,1,6,0,4,1,False,False,False,4,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138687,1529286,2,252,134,18,3,1,False,False,False,216,3,2026,2026-05-01,Jaipur,"Sawai Mansingh Stadium, Jaipur",134,134,252,bat,252,win,225
138688,1529286,2,252,134,18,4,1,False,False,False,217,3,2026,2026-05-01,Jaipur,"Sawai Mansingh Stadium, Jaipur",134,134,252,bat,252,win,225
138689,1529286,2,252,134,18,5,4,False,False,False,221,3,2026,2026-05-01,Jaipur,"Sawai Mansingh Stadium, Jaipur",134,134,252,bat,252,win,225
138690,1529286,2,252,134,19,0,4,False,False,False,225,3,2026,2026-05-01,Jaipur,"Sawai Mansingh Stadium, Jaipur",134,134,252,bat,252,win,225


In [9]:
second_innings_merged["target_runs"] = second_innings_merged["first_innings_runs"] + 1
second_innings_merged["runs_left"] = (
    second_innings_merged["target_runs"] - second_innings_merged["current_score"]
)

second_innings_merged

,match_id,innings,team_batting,team_bowling,over_number,ball_number,total_runs,is_wicket,is_wide_ball,is_no_ball,current_score,wickets_lost,season_id,match_date,city,venue,toss_winner,team1,team2,toss_decision,match_winner,result,first_innings_runs,target_runs,runs_left
0,335982,2,1,6,0,0,1,False,False,False,1,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,222
1,335982,2,1,6,0,1,1,False,True,False,2,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,221
2,335982,2,1,6,0,2,0,False,False,False,2,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,221
3,335982,2,1,6,0,3,1,False,False,False,3,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,220
4,335982,2,1,6,0,4,1,False,False,False,4,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,219
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138687,1529286,2,252,134,18,3,1,False,False,False,216,3,2026,2026-05-01,Jaipur,"Sawai Mansingh Stadium, Jaipur",134,134,252,bat,252,win,225,226,10
138688,1529286,2,252,134,18,4,1,False,False,False,217,3,2026,2026-05-01,Jaipur,"Sawai Mansingh Stadium, Jaipur",134,134,252,bat,252,win,225,226,9
138689,1529286,2,252,134,18,5,4,False,False,False,221,3,2026,2026-05-01,Jaipur,"Sawai Mansingh Stadium, Jaipur",134,134,252,bat,252,win,225,226,5
138690,1529286,2,252,134,19,0,4,False,False,False,225,3,2026,2026-05-01,Jaipur,"Sawai Mansingh Stadium, Jaipur",134,134,252,bat,252,win,225,226,1


Now comes a part i didn't like, we cannot simply use the number of rows, because a wide/no-ball doesn't count as a legal delivery. So we create another feature for indicating the legal balls

In [10]:
second_innings_merged["legal_ball"] = (
    ~second_innings_merged["is_wide_ball"] &
    ~second_innings_merged["is_no_ball"]
)

second_innings_merged["total_legal_balls"]=second_innings_merged.groupby("match_id")["legal_ball"].cumsum()
second_innings_merged.head()


,match_id,innings,team_batting,team_bowling,over_number,ball_number,total_runs,is_wicket,is_wide_ball,is_no_ball,current_score,wickets_lost,season_id,match_date,city,venue,toss_winner,team1,team2,toss_decision,match_winner,result,first_innings_runs,target_runs,runs_left,legal_ball,total_legal_balls
0,335982,2,1,6,0,0,1,False,False,False,1,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,222,True,1
1,335982,2,1,6,0,1,1,False,True,False,2,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,221,False,1
2,335982,2,1,6,0,2,0,False,False,False,2,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,221,True,2
3,335982,2,1,6,0,3,1,False,False,False,3,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,220,True,3
4,335982,2,1,6,0,4,1,False,False,False,4,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,219,True,4


CRR = Current score/Overs completed hence

In [11]:
second_innings_merged["overs_completed"] = (
    second_innings_merged["total_legal_balls"] / 6
)

second_innings_merged["CRR"] = (
    second_innings_merged["current_score"] /
    second_innings_merged["overs_completed"]
)

second_innings_merged

,match_id,innings,team_batting,team_bowling,over_number,ball_number,total_runs,is_wicket,is_wide_ball,is_no_ball,current_score,wickets_lost,season_id,match_date,city,venue,toss_winner,team1,team2,toss_decision,match_winner,result,first_innings_runs,target_runs,runs_left,legal_ball,total_legal_balls,overs_completed,CRR
0,335982,2,1,6,0,0,1,False,False,False,1,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,222,True,1,0.166667,6.000000
1,335982,2,1,6,0,1,1,False,True,False,2,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,221,False,1,0.166667,12.000000
2,335982,2,1,6,0,2,0,False,False,False,2,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,221,True,2,0.333333,6.000000
3,335982,2,1,6,0,3,1,False,False,False,3,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,220,True,3,0.500000,6.000000
4,335982,2,1,6,0,4,1,False,False,False,4,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,219,True,4,0.666667,6.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138687,1529286,2,252,134,18,3,1,False,False,False,216,3,2026,2026-05-01,Jaipur,"Sawai Mansingh Stadium, Jaipur",134,134,252,bat,252,win,225,226,10,True,112,18.666667,11.571429
138688,1529286,2,252,134,18,4,1,False,False,False,217,3,2026,2026-05-01,Jaipur,"Sawai Mansingh Stadium, Jaipur",134,134,252,bat,252,win,225,226,9,True,113,18.833333,11.522124
138689,1529286,2,252,134,18,5,4,False,False,False,221,3,2026,2026-05-01,Jaipur,"Sawai Mansingh Stadium, Jaipur",134,134,252,bat,252,win,225,226,5,True,114,19.000000,11.631579
138690,1529286,2,252,134,19,0,4,False,False,False,225,3,2026,2026-05-01,Jaipur,"Sawai Mansingh Stadium, Jaipur",134,134,252,bat,252,win,225,226,1,True,115,19.166667,11.739130


Now calculate the RRR=runs required.balls remaining

In [12]:
second_innings_merged["balls_left"] = (
    120 - second_innings_merged["total_legal_balls"]
)

second_innings_merged["overs_left"] = (
    second_innings_merged["balls_left"] / 6
)

second_innings_merged["RRR"] = (
    second_innings_merged["runs_left"] /
    second_innings_merged["overs_left"]
)

second_innings_merged

,match_id,innings,team_batting,team_bowling,over_number,ball_number,total_runs,is_wicket,is_wide_ball,is_no_ball,current_score,wickets_lost,season_id,match_date,city,venue,toss_winner,team1,team2,toss_decision,match_winner,result,first_innings_runs,target_runs,runs_left,legal_ball,total_legal_balls,overs_completed,CRR,balls_left,overs_left,RRR
0,335982,2,1,6,0,0,1,False,False,False,1,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,222,True,1,0.166667,6.000000,119,19.833333,11.193277
1,335982,2,1,6,0,1,1,False,True,False,2,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,221,False,1,0.166667,12.000000,119,19.833333,11.142857
2,335982,2,1,6,0,2,0,False,False,False,2,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,221,True,2,0.333333,6.000000,118,19.666667,11.237288
3,335982,2,1,6,0,3,1,False,False,False,3,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,220,True,3,0.500000,6.000000,117,19.500000,11.282051
4,335982,2,1,6,0,4,1,False,False,False,4,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,219,True,4,0.666667,6.000000,116,19.333333,11.327586
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138687,1529286,2,252,134,18,3,1,False,False,False,216,3,2026,2026-05-01,Jaipur,"Sawai Mansingh Stadium, Jaipur",134,134,252,bat,252,win,225,226,10,True,112,18.666667,11.571429,8,1.333333,7.500000
138688,1529286,2,252,134,18,4,1,False,False,False,217,3,2026,2026-05-01,Jaipur,"Sawai Mansingh Stadium, Jaipur",134,134,252,bat,252,win,225,226,9,True,113,18.833333,11.522124,7,1.166667,7.714286
138689,1529286,2,252,134,18,5,4,False,False,False,221,3,2026,2026-05-01,Jaipur,"Sawai Mansingh Stadium, Jaipur",134,134,252,bat,252,win,225,226,5,True,114,19.000000,11.631579,6,1.000000,5.000000
138690,1529286,2,252,134,19,0,4,False,False,False,225,3,2026,2026-05-01,Jaipur,"Sawai Mansingh Stadium, Jaipur",134,134,252,bat,252,win,225,226,1,True,115,19.166667,11.739130,5,0.833333,1.200000


Sanity check - If wickets_lost only increases when is_wicket is True

In [13]:
second_innings.loc[
    second_innings["is_wicket"] == True,
    ["match_id", "ball_number", "is_wicket", "wickets_lost"]
].head(20)

,match_id,ball_number,is_wicket,wickets_lost
131,335982,0,True,1
138,335982,1,True,2
154,335982,4,True,3
157,335982,1,True,4
174,335982,4,True,5
177,335982,1,True,6
183,335982,7,True,7
197,335982,0,True,8
210,335982,0,True,9
224,335982,1,True,10


Checking the max values of wickets lost

In [14]:
second_innings["wickets_lost"].max()

np.int64(10)

Saving the current dataframe into a dataset for further processing

In [15]:
second_innings_merged.to_csv("../data/processed/second_innings_merged.csv")